In [1]:
import networkx as nx
import numpy as np
import pandas as pd
from pecanpy import pecanpy as n2v
import random
from sklearn.linear_model import LogisticRegression
import ast
import pickle
from sklearn.metrics import roc_auc_score
from scipy.io import loadmat
import scipy.sparse as sp

Prepping Train/Test Split

In [ ]:
def split(G, frac=0.5):
    edges = list(G.edges())
    
    # guarantee fully connected training set by protecting spanning tree from removal
    spanning_tree_edges = set(nx.minimum_spanning_tree(G).edges())
    removable_edges = [e for e in edges if e not in spanning_tree_edges]

    # sample test edges from removable edges only
    test_count = int(len(edges) * frac)
    test_edges = set(random.sample(removable_edges, test_count))
    train_edges = [e for e in edges if e not in test_edges]
    
    # build training graph
    G_train = nx.Graph()
    G_train.add_nodes_from(G.nodes())
    G_train.add_edges_from(train_edges)
    
    # function to restrict sampled non-edges for better performance on large datasets
    def sample_non_edges(G, count):
        non_edges = set()
        nodes = list(G.nodes())
        while len(non_edges) < count:
            u, v = random.sample(nodes, 2)
            if not G.has_edge(u, v) and (u, v) not in non_edges:
                non_edges.add((u, v))
        return list(non_edges)

    # getting true negatives (overlap possible but very improbable for large datasets)
    test_non_edges = sample_non_edges(G, len(test_edges))
    train_non_edges = sample_non_edges(G, len(train_edges))
    
    return G_train, test_edges, test_non_edges, train_non_edges

# set filename

mat = True # handle matlab format
if mat:
    edgelist = loadmat('')
    adj_matrix = edgelist['network']
    G = nx.from_scipy_sparse_array(adj_matrix)
    print(f"Nodes: {G.number_of_nodes()}")
    print(f"Edges: {G.number_of_edges()}")
else:
    G = nx.read_edgelist('pipeline_files/test-data/ca-AstroPh.txt.gz')
    print(f"Nodes: {G.number_of_nodes()}")
    print(f"Edges: {G.number_of_edges()}")

# extracting lcc in case disconnected
largest_cc = max(nx.connected_components(G), key=len)
G = G.subgraph(largest_cc).copy()
G_train, test_edges, test_non_edges, train_non_edges = split(G)

Nodes: 3890
Edges: 38739


In [37]:
# saving training graph + edges
nx.write_edgelist(G_train, 'pipeline_files/ppi/train.txt')
lists = {
    'test_edges': test_edges,
    'test_non_edges': test_non_edges,
    'train_edges': list(G_train.edges()),
    'train_non_edges': train_non_edges
}
for name, l in lists.items():
    with open(f"pipeline_files/ppi/{name}.txt", 'w') as f:
        f.write(str(l))

Generating Embeddings

In [ ]:
g = n2v.PreComp(p=1, q=1, workers=4, verbose=True)

# load graph from edgelist file
g.read_edg('pipeline_files/ppi/train.txt', weighted=False, directed=False, delimiter=' ')
# precompute and save 2nd order transition probs (for PreComp only)
g.preprocess_transition_probs()

# generate random walks, which could then be used to train w2v
walks = g.simulate_walks(num_walks=10, walk_length=80)

# train embeddings using word2vec from gensim for more control
from gensim.models import Word2Vec
model = Word2Vec(walks, vector_size=128, window=10, min_count=0, 
                 sg=1, workers=4, epochs=1)

# saving (vectors only)
model.wv.save_word2vec_format("pipeline_files/ppi/vectors.txt", binary=False)

Training

In [39]:
# reading everything in
df = pd.read_csv('pipeline_files/ppi/vectors.txt', skiprows=1, sep=' ', header=None, index_col=0).copy()

with open("pipeline_files/ppi/train_edges.txt", "r") as f:
    train_edges = ast.literal_eval(f.read())
with open("pipeline_files/ppi/train_non_edges.txt", "r") as f:
    train_non_edges = ast.literal_eval(f.read())

df['embeddings'] = df.values.tolist() # combining embeddings into single column with lists
df.index = df.index.astype(str) # normalizing data structure
embedding_map = df['embeddings'].to_dict() # dict for fast lookup

# function to turn node embeddings into labeled edge embeddings
def create_training_data(edges, label, embedding_map=embedding_map):
    data = []
    for u, v in edges:
        # get embeddings for both nodes
        emb_u = embedding_map.get(str(u))
        emb_v = embedding_map.get(str(v))
        
        # store as (node_tuple, embedding_tuple, label)
        data.append({
            'edge': (u, v),
            'embeddings': (emb_u, emb_v),
            'label': label
        })
    return data

# generate data for both positive and negative samples
pos_data = create_training_data(G_train.edges(), 1)
neg_data = create_training_data(train_non_edges, 0)

# combine and create final df
training_set = pd.DataFrame(pos_data + neg_data)

# shuffle and reset index
training_set = training_set.sample(frac=1).reset_index(drop=True)

def train(training_set):
    # applying binary operators
    training_set['avg'] = training_set['embeddings'].apply(lambda x: np.mean(x, axis=0))
    training_set['hadamard'] = training_set['embeddings'].apply(lambda x: np.multiply(x[0], x[1]))
    training_set['w-l1'] = training_set['embeddings'].apply(lambda x: np.abs(np.subtract(x[0], x[1])))
    training_set['w-l2'] = training_set['embeddings'].apply(lambda x: np.square(np.subtract(x[0], x[1])))

    # training classifier for each operator
    operators = ['avg', 'hadamard', 'w-l1', 'w-l2']
    models = []
    for op in operators:
        X = np.stack(training_set[op].values)
        y = np.stack(training_set['label'].values)

        model = LogisticRegression(max_iter=1000)
        model.fit(X, y)

        models.append(model)
    
    # combined dict for lookup
    classifiers = dict(zip(operators, models))
    return classifiers

classifiers = train(training_set)

In [40]:
# save classifiers
with open("pipeline_files/ppi/classifiers.pkl", "wb") as f:
    pickle.dump(classifiers, f)

Testing

In [41]:
# same setup as training
df = pd.read_csv('pipeline_files/ppi/vectors.txt', skiprows=1, sep=' ', header=None, index_col=0).copy()

with open("pipeline_files/ppi/classifiers.pkl", 'rb') as f:
    classifiers = pickle.load(f)
with open("pipeline_files/ppi/test_edges.txt", "r") as f:
    test_edges = ast.literal_eval(f.read())
with open("pipeline_files/ppi/test_non_edges.txt", "r") as f:
    test_non_edges = ast.literal_eval(f.read())

df['embeddings'] = df.values.tolist() # combining embeddings into single column with lists
df.index = df.index.astype(str) # normalizing data structure
embedding_map = df['embeddings'].to_dict() # dict for fast lookup

# generate data for both positive and negative samples
pos_data = create_training_data(test_edges, 1)
neg_data = create_training_data(test_non_edges, 0)

# combine and create final df
testing_set = pd.DataFrame(pos_data + neg_data)

# shuffle and reset index
testing_set = testing_set.sample(frac=1).reset_index(drop=True)

def test(classifiers, testing_set):
    # applying binary operators as before
    testing_set['avg'] = testing_set['embeddings'].apply(lambda x: np.mean(x, axis=0))
    testing_set['hadamard'] = testing_set['embeddings'].apply(lambda x: np.multiply(x[0], x[1]))
    testing_set['w-l1'] = testing_set['embeddings'].apply(lambda x: np.abs(np.subtract(x[0], x[1])))
    testing_set['w-l2'] = testing_set['embeddings'].apply(lambda x: np.square(np.subtract(x[0], x[1])))

    # getting probabilities
    prob_dict = {}
    for op, model in classifiers.items():
        X_test = np.stack(testing_set[op].values)
        probs = model.predict_proba(X_test)[:,1]
        prob_dict[op] = probs

    # getting scores
    true_y = testing_set['label'].values
    auc_scores = {}
    for op in prob_dict.keys():
        prob = prob_dict[op]
        auc = roc_auc_score(true_y, prob)
        auc_scores[op] = auc

    return auc_scores

results = test(classifiers, testing_set)

for op, score in results.items():
    print(f"{op:10}: {score:.4f}")

avg       : 0.8140
hadamard  : 0.6900
w-l1      : 0.7335
w-l2      : 0.7365
